# Vuln2Action - evaluate target_steps epoch 1

The 5-epoch final checkpoint degraded badly. This notebook evaluates
`checkpoint-693` (epoch 1), which had the best eval loss, on 1,000 validation
rows for a fair comparison with the 1-epoch `target_techniques` result.


In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda devices:", torch.cuda.device_count())


In [ ]:
!rm -rf /kaggle/working/Vuln2Action-Lab
!git clone --branch feature/paper-reproduction https://github.com/Hoang-Minh-Hieu-2806/Vuln2Action-Lab.git /kaggle/working/Vuln2Action-Lab
%cd /kaggle/working/Vuln2Action-Lab
!pip install -q -e ".[train]"


In [ ]:
from pathlib import Path
import glob, os
hits=sorted(glob.glob("/kaggle/input/**/flow_train.jsonl", recursive=True))
print("flow_train hits:", hits)
if not hits: raise FileNotFoundError("flow_train.jsonl not found")
FLOW_DIR=Path(hits[0]).parent
os.environ["FLOW_DIR"]=str(FLOW_DIR)
os.environ["TOKENIZERS_PARALLELISM"]="false"
ckpts=sorted(glob.glob("/kaggle/input/notebooks/**/checkpoint-693", recursive=True))
print("checkpoint hits:", ckpts)
if not ckpts: raise FileNotFoundError("checkpoint-693 not found")
CKPT=ckpts[0]
os.environ["CKPT"]=CKPT
print("FLOW_DIR:", FLOW_DIR)
print("CKPT:", CKPT)


In [ ]:
import os, sys, subprocess
cmd=[
 sys.executable, "scripts/evaluate_t5.py",
 "--checkpoint", os.environ["CKPT"],
 "--dataset", os.environ["FLOW_DIR"],
 "--split", "validation",
 "--target-field", "target_steps",
 "--output-dir", "/kaggle/working/t5_stage1_target_steps_eval1",
 "--max-input-length", "256",
 "--max-target-length", "384",
 "--batch-size", "4",
 "--num-beams", "4",
 "--limit", "1000",
 "--fp16",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=os.environ.copy())


In [ ]:
import json
from pathlib import Path
out=Path("/kaggle/working/t5_stage1_target_steps_eval1")
metrics=json.loads((out/"metrics.json").read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2, ensure_ascii=False))
for i,line in enumerate((out/"predictions.jsonl").open(encoding="utf-8")):
    if i>=3: break
    row=json.loads(line)
    print(f"\n[{i}] {row['cve_id']}")
    print("REF:", row["reference"][:400])
    print("PRED:", row["prediction"][:400])
